# Neural Ordinary Differential Equations

Implementation walkthrough for [Neural ODEs](https://arxiv.org/abs/1806.07366) (Chen et al., NeurIPS 2018).

**Goal:** Build a Neural ODE from scratch to understand how continuous-depth models work.

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

In [ ]:
plt.rcParams.update({
    # remove top and right spines
    'axes.spines.top':    False,
    'axes.spines.right':  False,

    # softer spine + tick color
    'axes.edgecolor':     '#333333',
    'axes.linewidth':     0.8,
    'xtick.color':        '#333333',
    'ytick.color':        '#333333',
    'xtick.direction':    'out',
    'ytick.direction':    'out',

    # typography
    'font.family':        'sans-serif',
    'font.size':          11,
    'axes.titlesize':     12,
    'axes.titleweight':   'semibold',
    'axes.labelsize':     8,
    'axes.labelcolor':    '#333333',

    # # subtle grid
    # 'axes.grid':          True,
    # 'grid.color':         '#E5E5E5',
    # 'grid.linewidth':     0.6,
    # 'axes.axisbelow':     True,    # grid behind data, not over it

    # figure
    'figure.figsize':     (6, 4),
    'figure.dpi':         110,
    'savefig.dpi':        200,
    'savefig.bbox':       'tight',

    # cleaner lines
    'lines.linewidth':    1.8,
    'legend.frameon':     False,
})

from cycler import cycler
plt.rcParams['axes.prop_cycle'] = cycler('color',
    ['#4285F4',   #  Blue
    '#EA4335',   #  Red
    '#FBBC04',   #  Yellow
    '#34A853',   #  Green
    '#FF6D01',   # Orange
    '#46BDC6',   # Teal
    '#9334E6',   # Purple
    '#E91E63',]   # Pink
)


## 1. ODE Solvers


The core idea is that, instead of stacking discrete residual layers,

$$
\mathbf{h}_{t+1} = \mathbf{h}_t + f(\mathbf{h}_t, \theta_t),
$$

we can model the hidden state as evolving continuously in time and solve the ODE

$$
\frac{d\mathbf{h}(t)}{dt} = f(\mathbf{h}(t), t, \theta)
$$

with a numerical ODE solver.

We begin by introducing two simple numerical solvers, Euler and fourth-order Runge–Kutta (RK4), and use them to solve a simple linear ODE whose trajectories form a spiral with constant angular velocity:

$$
\mathbf{z} =
\begin{bmatrix}
x \\
y
\end{bmatrix}
$$

$$
\dot{\mathbf{z}} =
\begin{bmatrix}
-0.1 & 2.0 \\
-2.0 & -0.1
\end{bmatrix}
\mathbf{z}
$$

In [ ]:
#Implementation of Euler and RK4 ODE solvers

def euler_step(f, y, t, dt):
    """Single Euler step: y_{n+1} = y_n + dt * f(y_n, t_n) - each timestep dt add the derivative of the function at the current point"""
    return y + dt * f(y, t)


def rk4_step(f, y, t, dt):
    """Single RK4 step - idea is to approximate the derivative at multiple points and average them"""
    k1 = f(y, t)
    k2 = f(y + dt/2 * k1, t + dt/2)
    k3 = f(y + dt/2 * k2, t + dt/2)
    k4 = f(y + dt   * k3, t + dt)
    return y + (dt/6) * (k1 + 2*k2 + 2*k3 + k4)

def ode_solve(f, y0, t_span, n_steps=100, method='euler'):
    """Solve an ODE from t_span[0] to t_span[1].

    Args:
        f:       function f(y, t) returning dy/dt
        y0:      initial state (scalar or array)
        t_span:  (t_start, t_end)
        n_steps: number of integration steps
        method:  'euler' or 'rk4'

    Returns:
        ts: array of shape (n_steps + 1,)      — time points
        ys: array of shape (n_steps + 1, *y0.shape) — state at each time
    """
    step_fn = {'euler': euler_step, 'rk4': rk4_step}[method]

    t0, t1 = t_span
    dt = (t1 - t0) / n_steps

    y0 = torch.asarray(y0, dtype=float)
    ts = torch.linspace(t0, t1, n_steps + 1)
    ys = torch.empty((n_steps + 1,) + y0.shape)
    ys[0] = y0

    y, t = y0, t0
    for i in range(n_steps):
        y = step_fn(f, y, t, dt)
        t += dt
        ys[i + 1] = y

    return ts, ys


# Simple spiral ODE
def spiral_ode(z, t):
    """
    The dynamics dz/dt = f(z, t) for a 2D spiral.
    
    Args:
        z: state, array-like of shape (2,)  —  [x, y]
        t: time (scalar). Unused here since dynamics are time-invariant.
    
    Returns:
        dz/dt, array of shape (2,)
    """
    x, y = z[0], z[1]
    dxdt = -0.1 * x + 2.0 * y
    dydt = -2.0 * x - 0.1 * y
    return torch.tensor([dxdt, dydt])

Note the Euler's method accumulates errors while RK4 faithfully reconstructs the spiral!

In [ ]:
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
ts, ys = ode_solve(spiral_ode, torch.tensor([2.0, 0.0]), (0.0, 10.0),
                   n_steps=300, method='euler')
plt.plot(ys[:,0], ys[:,1], label='Euler', c='dodgerblue')
ts, ys = ode_solve(spiral_ode, torch.tensor([2.0, 0.0]), (0.0, 10.0),
                   n_steps=10000, method='rk4')
plt.plot(ys[:,0], ys[:,1], label='true', c='k', linestyle = '--', lw=1)

plt.gca().set_aspect('equal')
plt.legend(loc='center')

plt.subplot(1,2,2)

ts, ys = ode_solve(spiral_ode, torch.tensor([2.0, 0.0]), (0.0, 10.0),
                   n_steps=300, method='rk4')
plt.plot(ys[:,0], ys[:,1], label='RK4', c='darkorange')

ts, ys = ode_solve(spiral_ode, torch.tensor([2.0, 0.0]), (0.0, 10.0),
                   n_steps=10000, method='rk4')
plt.plot(ys[:,0], ys[:,1], label='true', c='k', linestyle = '--', lw=1)

plt.gca().set_aspect('equal')
plt.legend(loc='center')


There are even more efficient ODE adaptive solvers that could adaptively chose step size to keep error rate at a pre-defined level (*if the function doens't change much, you can take longer steps without the loss of precision*)

In [ ]:
def dopri5_step(f, y, t, dt):
    """One adaptive Dormand-Prince step. Returns (z_new, error_estimate)."""

    # Dormand-Prince 5(4) Butcher tableau coefficients# Dormand-Prince 5(4) Butcher tableau coefficients
    # (These are the standard constants — don't try to derive them, just copy.)
    _C = [0, 1/5, 3/10, 4/5, 8/9, 1.0, 1.0]
    _A = [
        [],
        [1/5],
        [3/40, 9/40],
        [44/45, -56/15, 32/9],
        [19372/6561, -25360/2187, 64448/6561, -212/729],
        [9017/3168, -355/33, 46732/5247, 49/176, -5103/18656],
        [35/384, 0, 500/1113, 125/192, -2187/6784, 11/84],
    ]
    _B5 = [35/384, 0, 500/1113, 125/192, -2187/6784, 11/84, 0]          # 5th order
    _B4 = [5179/57600, 0, 7571/16695, 393/640, -92097/339200, 187/2100, 1/40]  # 4th order

    k = []
    for i in range(7):
        y_stage = y.clone()
        for j, a in enumerate(_A[i]):
            y_stage = y_stage + dt * a * k[j]
        k.append(f(y_stage, t + _C[i] * dt))

    z5 = y + dt * sum(b * ki for b, ki in zip(_B5, k) if b != 0)
    z4 = y + dt * sum(b * ki for b, ki in zip(_B4, k) if b != 0)
    err = (z5 - z4).abs().max()
    return z5, err


def odeint_adaptive(f, y0, t, rtol=1e-5, atol=1e-7, dt_init=0.01):
    ys = [y0]
    y = y0
    dt = torch.tensor(dt_init)

    # Log EVERY accepted internal step, not just the requested t points
    internal_t  = [t[0].item() if torch.is_tensor(t[0]) else float(t[0])]
    internal_y  = [y0.clone()]
    internal_dt = []

    for i in range(len(t) - 1):
        t_cur, t_target = t[i], t[i + 1]
        
        # originally i put simply t_cur < t_target, but this is not dirrection
        # invariant which is critical (see chapter 3 to why). so will do a 
        # direction check instead
        direction = 1.0 if t_target > t_cur else -1.0
        dt = torch.tensor(dt_init) * direction

        while (t_cur - t_target ) * direction < 0:

            remaining = t_target - t_cur
            if abs(dt) > abs(remaining):
                dt = remaining

            y_new, err = dopri5_step(f, y, t_cur, dt)
            tol = atol + rtol * y.abs().max()

            if err <= tol:
                internal_dt.append(dt.item())
                y = y_new
                t_cur = t_cur + dt
                internal_t.append(t_cur.item())
                internal_y.append(y.clone())
                dt = dt * min(5.0, 0.9 * (tol / (err + 1e-12)) ** 0.2)
            else:
                dt = dt * max(0.1, 0.9 * (tol / err) ** 0.2)
        ys.append(y)

    return (
        torch.stack(ys, dim=0),               # state at requested t points
        torch.tensor(internal_t),             # every internal time the solver visited
        torch.stack(internal_y, dim=0),       # state at every internal step
        torch.tensor(internal_dt),            # the dt used for each accepted step
    )

Note that even with only a small number of points, the method recovers an accurate trajectory. The adaptive scheme selects step sizes automatically, allowing evaluation at specified times while maintaining the desired precision.

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)

ts, ys = ode_solve(spiral_ode, torch.tensor([2.0, 0.0]), (0.0, 10.0),
                   n_steps=15, method='rk4')
# plt.plot(ys[:,0], ys[:,1], c='darkorange', alpha=0.5)
plt.scatter(ys[:,0], ys[:,1], label='RK4', c='darkorange', s=50, marker='+')

ts, ys = ode_solve(spiral_ode, torch.tensor([2.0, 0.0]), (0.0, 10.0),
                   n_steps=10000, method='rk4')
plt.plot(ys[:,0], ys[:,1], label='true', c='k', linestyle = '--', lw=1)
plt.gca().set_aspect('equal')
plt.legend(loc='center')

plt.subplot(1,2,2)

ts = torch.linspace(0.0, 10.0, 15)
ys, t_int, y_int, dt_int = odeint_adaptive(spiral_ode, torch.tensor([2.0, 0.0]), ts, dt_init=2.)
# plt.plot(ys[:,0], ys[:,1], c='orangered', alpha=0.5)
plt.scatter(ys[:,0], ys[:,1], label='dopri5', c='orangered', s=50, marker='+')
plt.scatter(y_int[:,0], y_int[:,1], c='orangered', alpha=0.3, edgecolor='none', label='inter.')


ts, ys = ode_solve(spiral_ode, torch.tensor([2.0, 0.0]), (0.0, 10.0),
                   n_steps=10000, method='rk4')
plt.plot(ys[:,0], ys[:,1], label='true', c='k', linestyle = '--', lw=1)
plt.gca().set_aspect('equal')
plt.legend(loc='center')

To demonstated the adaptibility, let's plot <a href='https://en.wikipedia.org/wiki/Van_der_Pol_oscillator'>Van der Pol oscilator</a> dynamics which exhibits boundary layers requiring dense point sampling to keep the accuracy 

$$
\begin{aligned}
\frac{dx}{dt} &= v \\
\frac{dv}{dt} &= \mu \left(1 - x^2\right) v - x
\end{aligned}
$$

In [ ]:

def van_der_pol(y, t, mu=1.5):
    """Higher mu = stiffer = more dramatic adaptivity."""
    x, v = y[..., 0], y[..., 1]
    dxdt = v
    dvdt = mu * (1 - x**2) * v - x
    return torch.stack([dxdt, dvdt], dim=-1)

y0 = torch.tensor([2.0, 0.0])
t = torch.linspace(0, 10, 100)
ys = odeint_adaptive(van_der_pol, y0, t, dt_init=0.01)
ys, it, iy, idt = odeint_adaptive(van_der_pol, y0, t, dt_init=0.01)

plt.figure(figsize=(8, 4))


xs = torch.linspace(-5, 5, 20)
vs = torch.linspace(-3, 3, 20)
X, V = torch.meshgrid(xs, vs, indexing='ij')
grid = torch.stack([X, V], dim=-1)

dY = van_der_pol(grid, None)
DX = dY[..., 1].cpu().numpy()
DV = dY[..., 0].cpu().numpy()

mag = np.sqrt(DX**2 + DV**2) + 1e-8

# nonlinear scaling (square root compression)
scaled_mag = (mag) ** 0.3

DX_scaled = DX / mag * scaled_mag
DV_scaled = DV / mag * scaled_mag

# color by horizontal direction
vmax = np.abs(DX).max()

plt.quiver(
    X.cpu().numpy(),
    V.cpu().numpy(),
    DX_scaled, DV_scaled,
    DX,                      # color = left/right
    cmap='coolwarm',
    clim=(-vmax, vmax),
    units='dots',
    scale=0.15,
    width=2,     # shaft thickness
    headwidth=4,     # width of arrow head
    headlength=4,    # length of arrow head
    headaxislength=3 # head shape refinement

)

plt.plot(iy[:,1], iy[:,0], c='k', lw=1)
plt.scatter(iy[:,1], iy[:,0], s=10, c='k')
plt.gca().set_aspect('equal')
plt.gca().set_axis_off()
# plt.xlim(-5, 5)

## 2. Neural ODE and backpropagation

Alright, now we know how to solve arbitrary ODE and 

Define a neural network $f(\mathbf{h}, t, \theta)$ that parameterizes the dynamics,
and wrap it in an ODE solve to create a "continuous-depth" layer.

In [ ]:
# a rewrite of a ode solve as an ode integrator for the nn (returns point at a preefined time) 

FIXED_STEP_SOLVERS = {
    "euler": euler_step,
    "rk4": rk4_step,
}

ADAPTIVE_SOLVERS = {
    "dopri5": odeint_adaptive,
}


def _odeint_fixed(f, z0, t, method="rk4", n_steps_per_interval=100):
    """
    Fixed-step ODE integration over the time grid `t`.

    Args:
        f: callable f(z, t) -> dz/dt
        z0: initial state, shape (..., dim)
        t: 1D tensor of requested time points, shape (T,)
        method: 'euler' or 'rk4'
        n_steps_per_interval: number of internal steps between consecutive t points

    Returns:
        Tensor of shape (T, ..., dim)
    """
    if method not in FIXED_STEP_SOLVERS:
        raise ValueError(f"Unknown fixed-step solver '{method}'")

    step_fn = FIXED_STEP_SOLVERS[method]

    zs = [z0]
    z = z0

    for i in range(len(t) - 1):
        t0, t1 = t[i], t[i + 1]
        dt = (t1 - t0) / n_steps_per_interval
        t_cur = t0

        for _ in range(n_steps_per_interval):
            z = step_fn(f, z, t_cur, dt)
            t_cur = t_cur + dt

        zs.append(z)

    return torch.stack(zs, dim=0)

def odeint(
    f,
    z0,
    t,
    method="rk4",
    n_steps_per_interval=100,
    rtol=1e-5,
    atol=1e-7,
    dt_init=0.01,
    return_internal=False,
):
    """
    General ODE integrator that supports both fixed-step and adaptive solvers,
    and both tensor-valued and tuple-valued states. (see in chapter 3 why we need tuple solvers)
    """
    # --- Tuple-state path: flatten, recurse, split back ---
    if not torch.is_tensor(z0):
        shapes  = [zi.shape   for zi in z0]
        sizes   = [zi.numel() for zi in z0]
        offsets = [0]
        for n in sizes[:-1]:
            offsets.append(offsets[-1] + n)

        def flatten(parts):
            return torch.cat([p.reshape(-1) for p in parts], dim=0)

        def split(flat):
            return tuple(
                flat[off:off+n].view(s)
                for s, n, off in zip(shapes, sizes, offsets)
            )

        def f_flat(flat, t_):
            return flatten(f(split(flat), t_))

        flat_traj = odeint(
            f_flat, flatten(z0), t,
            method=method,
            n_steps_per_interval=n_steps_per_interval,
            rtol=rtol, atol=atol, dt_init=dt_init,
            return_internal=False,   # internal history doesn't make sense for tuples
        )
        # flat_traj has shape (T, total_dim). Split along the last axis into a tuple
        # of tensors with shapes (T, *shape_i).
        return tuple(
            flat_traj[:, off:off+n].reshape(flat_traj.shape[0], *s)
            for s, n, off in zip(shapes, sizes, offsets)
        )

    # --- Tensor-state path: your original dispatch, unchanged ---
    if method in FIXED_STEP_SOLVERS:
        return _odeint_fixed(
            f=f, z0=z0, t=t,
            method=method,
            n_steps_per_interval=n_steps_per_interval,
        )

    elif method in ADAPTIVE_SOLVERS:
        z_traj, internal_t, internal_y, internal_dt = ADAPTIVE_SOLVERS[method](
            f=f, y0=z0, t=t, rtol=rtol, atol=atol, dt_init=dt_init,
        )
        if return_internal:
            return z_traj, internal_t, internal_y, internal_dt
        return z_traj

# Define the dynamics network and Neural ODE layer

class ODEFunc(nn.Module):
    """The learned derivative dh/dt = f(h, t)."""
    def __init__(self, dim=2, hidden_dim=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, dim)
        )

    def forward(self, z, t):
        t_expanded = t * torch.ones_like(z[..., :1])
        return self.net(torch.cat([z, t_expanded], dim=-1))


class NeuralODEBlock(nn.Module):
    """Wraps ODEFunc in an ODE solver for forward pass via ."""
    def __init__(self, odefunc):
        super().__init__()
        self.odefunc = odefunc

    def forward(
        self,
        z0,
        t,
        solver="rk4",
        n_steps_per_interval=100,
        rtol=1e-5,
        atol=1e-7,
        dt_init=0.01,
        return_internal=False,
    ):
        return odeint(
            f=self.odefunc,
            z0=z0,
            t=t,
            method=solver,
            n_steps_per_interval=n_steps_per_interval,
            rtol=rtol,
            atol=atol,
            dt_init=dt_init,
            return_internal=return_internal,
        )

This is enough to train an Neural ODE. Let's create a dataset with the same simple spiral example and see if we can learn it! 

In [ ]:
def make_spiral_dataset(n_spirals=1000, n_steps=100, t_max=8.0, noise=0.):
    """
    Bi-directional spiral dataset from the Neural ODE paper.
    1000 spirals (1000 CW), each sampled at 100 equally-spaced timesteps,
    each starting at a different random point.
    """
    t = torch.linspace(0, t_max, n_steps)

    # Random starting points on a circle of radius ~1
    angles = torch.rand(n_spirals) * 2 * torch.pi
    r0 = 1.0
    x0 = r0 * torch.cos(angles)
    y0 = r0 * torch.sin(angles)
    z0 = torch.stack([x0, y0], dim=-1).to(torch.float64)  # (n_spirals, 2)

    # Spiral dynamics: dz/dt = A @ z
    # CW:  A = [[-0.1, -2.0], [2.0, -0.1]]
    # CCW: A = [[-0.1,  2.0], [-2.0, -0.1]]
    # labels = torch.zeros(n_spirals).long()
    # labels[n_spirals // 2:] = 1  # 1 = counter-clockwise

    trajs = torch.zeros(n_spirals, n_steps, 2)

    for i in range(n_spirals):
        sign = 1.0
        A = torch.tensor(
            [[-0.1, sign * 2.0],
             [-sign * 2.0, -0.1]],
            dtype=torch.float64
        )

        def f(z, t):
            return A @ z

        t, ys = ode_solve(f, z0[i], (0.0, t_max), n_steps=n_steps-1, method='rk4')
        trajs[i] = ys

    trajs += noise * torch.randn_like(trajs)
    return t, trajs


t, trajs = make_spiral_dataset()

plt.figure(figsize=(5, 5))
for i in np.random.randint(0, trajs.shape[0], 20):
    # c = 'C0' if labels[i] == 0 else 'C1'
    plt.plot(trajs[i, :, 0], trajs[i, :, 1], alpha=0.8)
plt.gca().set_aspect('equal')
plt.title(f'Bi-directional spirals')
plt.show()

In [ ]:
nODEFunc = ODEFunc()
nODE = NeuralODEBlock(nODEFunc)

adam = torch.optim.Adam(nODE.parameters(), lr=1e-2)

n_training_steps = 1000
batch_size = 16
n_samples = 50

pbar = tqdm(range(n_training_steps))

loss_history = []
for step in pbar:
    # sample batch of trajectories
    spiral_idx = torch.randperm(trajs.shape[0])[:batch_size]

    # sample time indices (sorted)
    t_idx = torch.randperm(trajs.shape[1])[:n_samples].sort().values
    # t_obs = t[t_idx]

    # initial state = first observed time in this subset
    z0 = trajs[spiral_idx][:, 0]          # (B, dim)

    # model prediction
    z_pred = nODE(z0, t, n_steps_per_interval=1)                     # (T, B, dim)

    # ground truth
    z_true = trajs[spiral_idx]              # (B, T, dim)

    # match shapes
    z_pred = z_pred.permute(1, 0, 2)             # (B, T, dim)

    # loss
    loss = (z_pred[:,t_idx] - z_true[:, t_idx]).pow(2).mean()

    adam.zero_grad()
    loss.backward()
    adam.step()

    # update tqdm
    pbar.set_postfix({
        "loss": f"{loss.item():.6f}"
    })
    loss_history.append(loss.item())

In [ ]:

plt.plot(loss_history)
plt.title('Loss');

Let's evaluate on 4 unseen curves started within the same circle of radius one, but this time we are going to extend predictions beyond the observed horizon (green), to see if we managed to generalise. And we did! 

In [ ]:
# ---- generate 4 unseen initial points ----
n_test = 4
t_max_train = 8.0
t_max_test = t_max_train + 10.0
n_steps_test = 200

angles = torch.rand(n_test) * 2 * torch.pi
r0 = 1.0
x0 = r0 * torch.cos(angles)
y0 = r0 * torch.sin(angles)
z0_test = torch.stack([x0, y0], dim=-1).to(torch.float64)   # (4, 2)

# ---- make true trajectories up to t_max + 5 using RK4 ----
def generate_from_z0(z0, n_steps=200, t_max=13.0, sign=1.0):
    t = torch.linspace(0, t_max, n_steps, dtype=torch.float64)
    trajs = torch.zeros(z0.shape[0], n_steps, 2, dtype=torch.float64)

    A = torch.tensor([[-0.1, sign * 2.0],
                      [-sign * 2.0, -0.1]], dtype=torch.float64)

    def f(z, tt):
        return A @ z

    for i in range(z0.shape[0]):
        z = z0[i].clone()
        trajs[i, 0] = z

        for j in range(n_steps - 1):
            dt = t[j + 1] - t[j]
            z = rk4_step(f, z, t[j], dt)
            trajs[i, j + 1] = z

    return t, trajs

t_test, trajs_test = generate_from_z0(
    z0_test, n_steps=n_steps_test, t_max=t_max_test
)

# ---- run model on same initial points up to t_max + 5 ----
nODE.eval()
device = next(nODE.parameters()).device

with torch.no_grad():
    z_pred = nODE(
        z0_test.to(device).to(torch.float32),   # (4, 2)
        t_test.to(device).to(torch.float32),    # (T,)
        solver="rk4",
        n_steps_per_interval=50
    )                                           # (T, 4, 2)

z_pred = z_pred.permute(1, 0, 2).cpu().to(torch.float64)   # (4, T, 2)

# ---- plot 4 curves ----
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
axes = axes.flatten()

obs_mask = t_test <= t_max_train

for i, ax in enumerate(axes):
    z_true = trajs_test[i]
    z_hat = z_pred[i]

    # full true trajectory to t_max + 5
    ax.plot(z_true[:, 0], z_true[:, 1],
            color='black', lw=2, alpha=0.7, label='true full')

    # observed part, only within training horizon
    ax.plot(z_true[obs_mask, 0], z_true[obs_mask, 1],
            color='green', lw=3, alpha=0.9, label='true observed', zorder=5)

    # model prediction on full horizon
    ax.plot(z_hat[:, 0], z_hat[:, 1],
            color='red', lw=2, alpha=0.85, label='pred')
    ax.set_axis_off()

    # # initial point
    # ax.scatter(z_true[0, 0], z_true[0, 1], color='blue', s=30)

    # # boundary point at t_max_train
    # boundary_idx = torch.where(obs_mask)[0][-1].item()
    # ax.scatter(z_true[boundary_idx, 0], z_true[boundary_idx, 1],
    #            color='orange', s=35, zorder=6)

    ax.set_aspect('equal')
    ax.set_title(f'curve {i+1}')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right')
plt.tight_layout()
plt.show()

## 3. Adjoint Method

Alright, all this was fairly simple. However, quoting the paper:

> The main technical difficulty in training continuous-depth networks is performing reverse-mode
> differentiation (also known as backpropagation) through the ODE solver. Differentiating through
> the operations of the forward pass is straightforward, but incurs a **high memory cost** and introduces
> additional numerical error.

The key contribution of the paper is finding a way to compute gradients **without** backpropagating through the forward pass of the solver. Instead, they compute gradients by running another augmented ODE **backwards in time** (see Figure 2). This lets us ignore the original computational graph and save a lot on memory.

### The algorithm in brief

**Input:** dynamics parameters $\theta$, start time $t_0$, stop time $t_1$, final state $z(t_1)$, loss gradient $\partial L / \partial z(t_1)$.

**Step 1: Build the augmented state.**
$$s_0 = [\,z(t_1),\; \partial L / \partial z(t_1),\; 0_{|\theta|}\,]$$
The third slot starts at $0$ and will accumulate $\partial L / \partial \theta$ as we solve the first two integrals backwards in time.

**Step 2: Define the augmented dynamics.**
$$f_{\text{aug}}([z(t), a(t), \cdot], t, \theta) = \big[\,f(z(t), t, \theta),\; -a(t)^T \tfrac{\partial f}{\partial z},\; -a(t)^T \tfrac{\partial f}{\partial \theta}\,\big]$$
The first slot is the original ODE. The second is the adjoint ODE (equation 4). The third is equation 5. Reminder: $a(t) = \partial L / \partial z(t)$.

**Step 3: Solve backwards in time.**
$$[\,z(t_0),\; \partial L / \partial z(t_0),\; \partial L / \partial \theta\,] = \text{ODESolve}(s_0,\, f_{\text{aug}},\, t_1,\, t_0,\, \theta)$$

**Step 4: Profit.**
- $z(t_0)$ - should match the original input (sanity check).
- $\partial L / \partial z(t_0)$ - hand back to whatever layers sit before the nODE block.
- $\partial L / \partial \theta$ - feeds into the optimizer.

If you have several observation times, apply this algorithm segment by segment, adding the loss gradient at each observation to the adjoint (see Figure 2, bottom).

We reuse the previously defined `odeint` function as our black-box ODE solver — no gradients will be passed through it.

In [ ]:
from typing import Any


class ODEAdjoint(torch.autograd.Function):
    """
    Differentiable ODE solve via the adjoint method.
    Forward: black-box solve, no autograd graph retained -> O(1) memory.
    Backward: augmented reverse ODE -> dL/dz0 and dL/dtheta.
    """

    @staticmethod
    def forward(ctx, odefunc, z0, t, solver_kwargs, *theta):
        with torch.no_grad():   # no gradients!!! 
            z_traj = odeint(odefunc, z0, t, **solver_kwargs)

        ctx.odefunc = odefunc
        ctx.solver_kwargs = solver_kwargs
        ctx.save_for_backward(t, z_traj, *theta) 
        return z_traj

    @staticmethod
    def backward(ctx, grad_z_traj):
        odefunc = ctx.odefunc
        solver_kwargs = ctx.solver_kwargs
        t, z_traj, *theta = ctx.saved_tensors
        T = z_traj.shape[0]

        # STEP 1: initial augmented state
        z_i = z_traj[-1]
        a_i = grad_z_traj[-1] # grad_z_traj = dL/dz(t1)
        g_i = tuple(torch.zeros_like(p) for p in theta)


        # STEP 2: augmented dynamics f_aug([z, a, g_theta], t) 
        def aug_dynamics(aug, t_scalar):
            z, a, *_ = aug  #[z, a, g_theta]
            with torch.enable_grad():
                z_ = z.detach().requires_grad_(True)  # grads only thorugh local segment
                f_val = odefunc(z_, t_scalar)  # f(z, t)
                grads = torch.autograd.grad(  # aT * df/dz , aT * df/dtheta
                    f_val, (z_, *theta), grad_outputs=a, allow_unused=True,
                )
            vjp_z, vjp_theta = grads[0], grads[1:] # these are our vector jacobian products
            vjp_theta = tuple[Any, ...]( # fails if not safeguard against the parameters that don't impact gradient, not important for understaning 
                g if g is not None else torch.zeros_like(p)
                for g, p in zip(vjp_theta, theta)
            )
            return (f_val.detach(), -vjp_z, *(-g for g in vjp_theta))


        # STEP 3: solve backwards, jump at each observation 
        for i in range(T - 1, 0, -1):
            aug_i = (z_i, a_i, *g_i)
            t_segment = torch.stack([t[i], t[i - 1]])
            aug_prev = odeint(aug_dynamics, aug_i, t_segment, **solver_kwargs)
            z_i, a_i, *g_list = (x[-1] for x in aug_prev)
            g_i = tuple(g_list)
            a_i = a_i + grad_z_traj[i - 1]

        # STEP 4: return grads matching forward's inputs 
        # (odefunc, z0, t, solver_kwargs, *params)
        return (None, a_i, None, None, *g_i)

Great, now we just slightly change the `NeuralODEBlock` to support the adjoint method and it's done

In [ ]:
class NeuralODEBlock(nn.Module):
    """Wraps ODEFunc in an ODE solver for forward pass via an ODE solver."""
    def __init__(self, odefunc):
        super().__init__()
        self.odefunc = odefunc

    def forward(
        self,
        z0,
        t,
        solver="rk4",
        n_steps_per_interval=100,
        rtol=1e-5,
        atol=1e-7,
        dt_init=0.01,
        return_internal=False,
        adjoint=False,
    ):
        solver_kwargs = dict(
            method=solver,
            n_steps_per_interval=n_steps_per_interval,
            rtol=rtol,
            atol=atol,
            dt_init=dt_init,
        )

        if adjoint:
            # O(1)-memory path: route through the adjoint autograd.Function.
            return ODEAdjoint.apply(
                self.odefunc, z0, t, solver_kwargs, *self.odefunc.parameters(),
            )
        else:
            # Standard path: backprop straight through the solver.
            return odeint(
                f=self.odefunc,
                z0=z0,
                t=t,
                return_internal=return_internal,
                **solver_kwargs,
            )

Let's see if it wokrs

In [ ]:
nODEFunc = ODEFunc()
nODE = NeuralODEBlock(nODEFunc)

adam = torch.optim.Adam(nODE.parameters(), lr=1e-2)

n_training_steps = 1000
batch_size = 16
n_samples = 50

pbar = tqdm(range(n_training_steps))

loss_history = []
for step in pbar:
    # sample batch of trajectories
    spiral_idx = torch.randperm(trajs.shape[0])[:batch_size]

    # sample time indices (sorted)
    t_idx = torch.randperm(trajs.shape[1])[:n_samples].sort().values
    # t_obs = t[t_idx]

    # initial state = first observed time in this subset
    z0 = trajs[spiral_idx][:, 0]          # (B, dim)

    # model prediction
    z_pred = nODE(z0, t, solver="dopri5", adjoint=True, n_steps_per_interval=1)                     # (T, B, dim)

    # ground truth
    z_true = trajs[spiral_idx]              # (B, T, dim)

    # match shapes
    z_pred = z_pred.permute(1, 0, 2)             # (B, T, dim)

    # loss
    loss = (z_pred[:,t_idx] - z_true[:, t_idx]).pow(2).mean()

    adam.zero_grad()
    loss.backward()
    adam.step()

    # update tqdm
    pbar.set_postfix({
        "loss": f"{loss.item():.6f}"
    })
    loss_history.append(loss.item())

In [ ]:
plt.plot(loss_history)
plt.title('Loss');

In [ ]:
# ---- generate 4 unseen initial points ----
n_test = 4
t_max_train = 8.0
t_max_test = t_max_train + 10.0
n_steps_test = 200

angles = torch.rand(n_test) * 2 * torch.pi
r0 = 1.0
x0 = r0 * torch.cos(angles)
y0 = r0 * torch.sin(angles)
z0_test = torch.stack([x0, y0], dim=-1).to(torch.float64)   # (4, 2)

# ---- make true trajectories up to t_max + 5 using RK4 ----
def generate_from_z0(z0, n_steps=200, t_max=13.0, sign=1.0):
    t = torch.linspace(0, t_max, n_steps, dtype=torch.float64)
    trajs = torch.zeros(z0.shape[0], n_steps, 2, dtype=torch.float64)

    A = torch.tensor([[-0.1, sign * 2.0],
                      [-sign * 2.0, -0.1]], dtype=torch.float64)

    def f(z, tt):
        return A @ z

    for i in range(z0.shape[0]):
        z = z0[i].clone()
        trajs[i, 0] = z

        for j in range(n_steps - 1):
            dt = t[j + 1] - t[j]
            z = rk4_step(f, z, t[j], dt)
            trajs[i, j + 1] = z

    return t, trajs

t_test, trajs_test = generate_from_z0(
    z0_test, n_steps=n_steps_test, t_max=t_max_test
)

# ---- run model on same initial points up to t_max + 5 ----
nODE.eval()
device = next(nODE.parameters()).device

with torch.no_grad():
    z_pred = nODE(
        z0_test.to(device).to(torch.float32),   # (4, 2)
        t_test.to(device).to(torch.float32),    # (T,)
        solver="rk4",
        n_steps_per_interval=50
    )                                           # (T, 4, 2)

z_pred = z_pred.permute(1, 0, 2).cpu().to(torch.float64)   # (4, T, 2)

# ---- plot 4 curves ----
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
axes = axes.flatten()

obs_mask = t_test <= t_max_train

for i, ax in enumerate(axes):
    z_true = trajs_test[i]
    z_hat = z_pred[i]

    # full true trajectory to t_max + 5
    ax.plot(z_true[:, 0], z_true[:, 1],
            color='black', lw=2, alpha=0.7, label='true full')

    # observed part, only within training horizon
    ax.plot(z_true[obs_mask, 0], z_true[obs_mask, 1],
            color='green', lw=3, alpha=0.9, label='true observed', zorder=5)

    # model prediction on full horizon
    ax.plot(z_hat[:, 0], z_hat[:, 1],
            color='red', lw=2, alpha=0.85, label='pred')
    ax.set_axis_off()

    # # initial point
    # ax.scatter(z_true[0, 0], z_true[0, 1], color='blue', s=30)

    # # boundary point at t_max_train
    # boundary_idx = torch.where(obs_mask)[0][-1].item()
    # ax.scatter(z_true[boundary_idx, 0], z_true[boundary_idx, 1],
    #            color='orange', s=35, zorder=6)

    ax.set_aspect('equal')
    ax.set_title(f'curve {i+1}')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right')
plt.tight_layout()
plt.show()

## 4. Replacing ResNet blocks with a single Neural ODE block

Cool — we can train arbitrary continuous dynamics instead of stacking bulky ResNet blocks. Essentially what they did in the paper is replace a classical ResNet like this:

```
image (28×28×1)
    ↓  [conv, downsample]
    ↓  [conv, downsample]
feature map h₀  (e.g. 7×7×64)
    ↓  residual block 1:  h₁ = h₀ + f₁(h₀)
    ↓  residual block 2:  h₂ = h₁ + f₂(h₁)
    ↓  ... 6 blocks total
feature map h₆
    ↓  [pool, linear]
logits  (10-dim)
    ↓  cross-entropy loss
```

with this:

```
image (28×28×1)
    ↓  [conv, downsample]                    ← same as ResNet
    ↓  [conv, downsample]                    ← same as ResNet
feature map z(t₀)  (7×7×64)                  ← initial condition z₀
    ↓  ODESolve(z(t₀), f, t₀=0, t₁=1, θ)     ← adaptive solvers can automatically
feature map z(t₁)  (7×7×64)                     change the effective *depth*
    ↓  [pool, linear]                        ← same as ResNet
logits
    ↓  cross-entropy loss
```

In [ ]:
device = "mps" if torch.mps.is_available() else "cpu"

def stem(): return nn.Sequential(
    nn.Conv2d(1, 64, 3, stride=2, padding=1), nn.GroupNorm(32, 64), nn.ReLU(),
    nn.Conv2d(64, 64, 3, stride=2, padding=1), nn.GroupNorm(32, 64), nn.ReLU(),
)  # (B,1,28,28) -> (B,64,7,7)

def head(): return nn.Sequential(
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10),
)  # (B,64,7,7) -> (B,10)

# ---------- ResNet ----------
class ResBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(64, 64, 3, padding=1); self.n1 = nn.GroupNorm(32, 64)
        self.c2 = nn.Conv2d(64, 64, 3, padding=1); self.n2 = nn.GroupNorm(32, 64)
    def forward(self, h):
        return h + self.n2(self.c2(F.relu(self.n1(self.c1(h)))))

class ResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem   = stem()
        self.blocks = nn.Sequential(*[ResBlock() for _ in range(6)])
        self.head   = head()
    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))

# ---------- ODE-Net ----------
class ConvODEFunc(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(65, 64, 3, padding=1); self.n1 = nn.GroupNorm(32, 64)
        self.c2 = nn.Conv2d(64, 64, 3, padding=1); self.n2 = nn.GroupNorm(32, 64)
    def forward(self, z, t):
        B, _, H, W = z.shape
        t_ch = t * torch.ones(B, 1, H, W, device=z.device, dtype=z.dtype)
        return self.n2(self.c2(F.relu(self.n1(self.c1(torch.cat([z, t_ch], 1))))))

class ODENet(nn.Module):
    def __init__(self, adjoint=True):
        super().__init__()
        self.stem    = stem()
        self.odeblock = NeuralODEBlock(ConvODEFunc())
        self.head    = head()
        self.adjoint = adjoint
        self.register_buffer("t", torch.tensor([0.0, 1.0]))
    def forward(self, x):
        z0 = self.stem(x)
        z1 = self.odeblock(z0, self.t, solver="rk4",
                           n_steps_per_interval=4, adjoint=self.adjoint)[-1]
        return self.head(z1)

In [ ]:
tfm = transforms.Compose([transforms.ToTensor(),
                          transforms.Normalize((0.1307,), (0.3081,))])
train_ds = datasets.MNIST("./data", train=True,  download=True, transform=tfm)
test_ds  = datasets.MNIST("./data", train=False, download=True, transform=tfm)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=512)

In [ ]:
from typing import Any


@torch.no_grad()
def accuracy(model):
    model.eval()
    correct = total = 0
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total   += y.size(0)
    return correct / total

results = {}
for name, model in [("ResNet", ResNet()), ("ODE-Net", ODENet(adjoint=True))]:
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6

    model.train()
    print(f'training {name} with {n_params}M params')
    for x, y in tqdm(train_loader):                           # one epoch
        x, y = x.to(device), y.to(device)
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()

    acc = accuracy(model)
    results[name] = (acc, n_params)
    print(f"{name:8s}  test acc = {acc:.4f}   params = {n_params:.3f}M")

# ---------- final comparison ----------
print("\n=== comparison ===")
for name, (acc, n) in results.items():
    print(f"{name:8s}  acc = {acc*100:5.2f}%   params = {n:.3f}M")

## 5. Continuous Normalizing Flows (don't have time)

Neural ODEs enable normalizing flows that don't require partitioning the dimensions.
The change in log-density is given by the trace of the Jacobian:

$$\frac{\partial \log p(\mathbf{z}(t))}{\partial t} = -\text{Tr}\left(\frac{\partial f}{\partial \mathbf{z}(t)}\right)$$

## 6. Laten ODEs

The last epxeriment of the paper 

```
observations x(t₀), x(t₁), ..., x(t_N)
        ↓  RNN encoder (runs BACKWARDS through the sequence)
h₀, h₁, ..., h_N  (RNN hidden states)
        ↓  final hidden state -> linear
μ, σ  (posterior over z(t₀))
        ↓  reparameterized sample
z(t₀)   ← latent initial state (4-dim)
        ↓  ODESolve with learned dynamics f(z, t, θ_f)
z(t₀), z(t₁), ..., z(t_N), z(t_{N+1}), ..., z(t_M)
        ↓  decoder (small MLP): z(t_i) -> x̂(t_i)
reconstructions x̂(t_i)

```

In [ ]:
device = "mps" if torch.mps.is_available() else "cpu"

# ---------- data: bidirectional noisy spirals ----------
def make_spiral_dataset(n_spirals=1000, n_points=100, noise=0.05):
    t = torch.linspace(0, 6*np.pi, n_points)
    trajs = []
    trajs_noise = []
    labels = []
    for _ in range(n_spirals):
        direction = 1.0 if torch.rand(1).item() < 0.5 else -1.0
        r0    = 0.5 + torch.rand(1).item()                         # start radius
        phase = 2*np.pi * torch.rand(1).item()                     # random rotation
        r = r0 * torch.exp(-0.1 * t)                                # shrink over time
        theta = direction * t + phase
        xy = torch.stack([r*torch.cos(theta), r*torch.sin(theta)], dim=-1)
        xy_noise = xy + noise * torch.randn_like(xy)
        trajs.append(xy)
        trajs_noise.append(xy_noise)
        labels.append(direction)
    trajs = torch.stack(trajs, dim=0)                               # (N, T, 2)
    trajs_noise = torch.stack(trajs_noise, dim=0) 
    return trajs, trajs_noise, t, labels

trajs, trajs_noise, t, labels = make_spiral_dataset()
trajs, trajs_noise, t = trajs.to(device), trajs_noise.to(device), t.to(device)

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(trajs[0][:,0].detach().cpu(), trajs[0][:,1].detach().cpu(), c='k')
plt.scatter(trajs_noise[0][:,0].detach().cpu(), trajs_noise[0][:,1].detach().cpu(), zorder=3)
plt.gca().set_aspect('equal')
plt.title(labels[0])

plt.subplot(1,2,2)
plt.plot(trajs[3][:,0].detach().cpu(), trajs[3][:,1].detach().cpu(), c='k')
plt.scatter(trajs_noise[3][:,0].detach().cpu(), trajs_noise[3][:,1].detach().cpu(), zorder=3)
plt.gca().set_aspect('equal')
plt.title(labels[3])
plt.tight_layout()

In [ ]:
class LatentODE(nn.Module):
    def __init__(self, data_dim=2, latent_dim=4, rnn_hidden=25, f_hidden=20, dec_hidden=20):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder: RNN run backwards over the sequence, then project to (mu, logvar)
        self.rnn = nn.GRU(input_size=data_dim + 1,   # +1 for time
                          hidden_size=rnn_hidden, batch_first=True)
        self.to_latent = nn.Linear(rnn_hidden, 2 * latent_dim)

        # Dynamics f(z, t, theta_f) -- wrapped in our NeuralODEBlock
        class ODEFunc(nn.Module):
            def __init__(self):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(latent_dim + 1, f_hidden), nn.Tanh(),
                    nn.Linear(f_hidden, latent_dim),
                )
            def forward(self, z, t):
                t_ch = t * torch.ones_like(z[..., :1])
                return self.net(torch.cat([z, t_ch], dim=-1))
        self.odeblock = NeuralODEBlock(ODEFunc())

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, dec_hidden), nn.ReLU(),
            nn.Linear(dec_hidden, data_dim),
        )

    def encode(self, x, t_x):
        # Concatenate time to each observation, run RNN backwards.
        B, T, _ = x.shape
        t_ch = t_x.view(1, T, 1).expand(B, T, 1)
        inp  = torch.cat([x, t_ch], dim=-1)
        inp_reversed = torch.flip(inp, dims=[1])                    # backwards in time
        _, h = self.rnn(inp_reversed)                               # h: (1, B, rnn_hidden)
        params = self.to_latent(h.squeeze(0))                       # (B, 2*latent_dim)
        mu, logvar = params.chunk(2, dim=-1)
        return mu, logvar

    def forward(self, x_obs, t_obs, t_full):
        """
        x_obs:  (B, N_obs, 2)   observations
        t_obs:  (N_obs,)        their timestamps (used only by encoder)
        t_full: (T,)            timestamps at which we want z and x_hat
                                (can include points beyond t_obs for extrapolation)
        """
        mu, logvar = self.encode(x_obs, t_obs)
        std = (0.5 * logvar).exp()
        z0  = mu + std * torch.randn_like(std)                      # reparameterize

        # ODE solve over the full time grid
        z_traj = self.odeblock(z0, t_full, solver="rk4",
                               n_steps_per_interval=2, adjoint=True) # (T, B, latent_dim)
        x_hat  = self.decoder(z_traj).permute(1, 0, 2)               # (B, T, 2)
        return x_hat, mu, logvar

def elbo_loss(x_hat_at_obs, x_obs, mu, logvar, noise_std=0.05):
    # Gaussian log-likelihood (fixed observation noise) + KL to N(0, I)
    recon = 0.5 * ((x_hat_at_obs - x_obs) / noise_std).pow(2).sum(dim=(1, 2))
    kl    = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=-1)
    return (recon + kl).mean()


In [ ]:
model = LatentODE().to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-2)

n_steps    = 1000
batch_size = 64
n_obs      = 30          # Section 5.1: "30/100" setting

loss_history = []
pbar = tqdm(range(n_steps))

for step in pbar:
    idx   = torch.randperm(trajs_noise.shape[0])[:batch_size]
    obs_i = torch.randperm(trajs_noise.shape[1])[:n_obs].sort().values
    x_obs = trajs_noise[idx][:, obs_i]                                    # (B, n_obs, 2)
    t_obs = t[obs_i]

    x_hat, mu, logvar = model(x_obs, t_obs, t)                      # (B, T, 2)
    x_hat_at_obs = x_hat[:, obs_i]
    loss = elbo_loss(x_hat_at_obs, x_obs, mu, logvar)

    opt.zero_grad(); loss.backward(); opt.step()
    loss_history.append(loss.item())
    pbar.set_postfix({
        "elbo": f"{loss.item():.6f}"
        })



In [ ]:
plt.plot(loss_history);

In [ ]:
model.eval()
traj_ids = [0, 1, 2, 3]
with torch.no_grad():
    # Take the first 4 spirals; observe only first 30% of each, extrapolate the rest.
    x = trajs_noise[traj_ids]
    cutoff = int(0.3 * len(t))
    obs_i  = torch.randperm(cutoff)[:n_obs].sort().values
    x_obs  = x[:, obs_i]
    t_obs  = t[obs_i]

    # Build a time grid: observations are in [0, t[cutoff]],
    # and we want predictions across the full interval for plotting.
    x_hat, _, _ = model(x_obs, t_obs, t)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for k, ax in enumerate(axes):
    ax.plot(trajs[traj_ids[k], :, 0].cpu(), trajs[traj_ids[k], :, 1].cpu(),
            "g-", alpha=0.5, label="ground truth")
    ax.plot(x_hat[k, :cutoff, 0].cpu(), x_hat[k, :cutoff, 1].cpu(),
            "b-", label="reconstruction")
    ax.plot(x_hat[k, cutoff:, 0].cpu(), x_hat[k, cutoff:, 1].cpu(),
            "r-", label="extrapolation")
    ax.scatter(x_obs[k, :, 0].cpu(), x_obs[k, :, 1].cpu(),
               s=15, c="k", label="observations")
    ax.set_aspect("equal"); ax.set_title(f"spiral {k}")
    if k == 0: ax.legend(loc="best", fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
import plotly.graph_objects as go

with torch.no_grad():
    # Encode 300 spirals to get posterior means z(t_0).
    mu, _ = model.encode(trajs[:300], t)                         # (300, 4)

    # Integrate the learned ODE from each mu over the full time grid.
    z_traj = model.odeblock(
        mu, t, solver="rk4", n_steps_per_interval=2, adjoint=False
    )                                                             # (T, 300, 4)
    z_traj = z_traj.permute(1, 0, 2).detach().cpu().numpy()       # (300, T, 4)

    # Take the first three latent dimensions for 3D view.
    z_traj_3d = z_traj[..., :4]                                   # (300, T, 3)
    direction = np.array(labels)[:300]

# Plotly is interactive in VS Code/Cursor notebooks (rotate/zoom/pan).
fig = go.Figure()
for i in range(z_traj_3d.shape[0]):
    color = "rgba(31,60,180,0.35)" if direction[i] > 0 else "rgba(214,39,40,0.35)"
    fig.add_trace(go.Scatter3d(
        x=z_traj_3d[i, :, 0],
        y=z_traj_3d[i, :, 1],
        z=z_traj_3d[i, :, 2],
        mode="lines",
        line=dict(color=color, width=2),
        showlegend=False,
        hoverinfo="skip"
    ))

fig.update_layout(
    title="Latent ODE trajectories (dims 0, 1, 2)",
    scene=dict(
        xaxis_title="latent dim 0",
        yaxis_title="latent dim 1",
        zaxis_title="latent dim 2",
    ),
    width=400,
    height=400,
    margin=dict(l=0, r=0, t=50, b=0),
)
fig.show()

In [ ]:
%pip install plotly
